# Lesson 12 - 使用 Agent Scratchpad 進行聊天記錄縮減

此筆記本展示如何使用 Microsoft Agent Framework 管理長對話中的上下文。隨著對話增長，令牌數量增加——最終超出模型的上下文窗口。我們透過**上下文摘要模式**和用於持久記憶的**agent scratchpad**來解決此問題。

## 您將學到：
1. **為何上下文管理重要**：了解令牌限制與上下文窗口
2. **具上下文感知的代理**：構建能管理自身對話上下文的代理
3. **上下文摘要模式**：使用工具濃縮對話歷史
4. **Agent Scratchpad**：在上下文縮減後仍能存活的持久記憶

## 前置條件：
- 已設定 Azure OpenAI 並配置環境變數
- 理解先前課程的基本代理概念


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 為何上下文管理很重要

每個大型語言模型都有有限的**上下文視窗**——一次請求中可處理的最大標記數量。隨著多輪對話進行：

- **標記數量隨每則用戶訊息和助理回覆線性增加**。
- **提示標記占成本主導**，因為每輪都會重新傳送整個歷史。
- 最終對話會**超出上下文視窗**，模型要麼截斷，要麼出錯。

### 管理上下文的策略

| 策略 | 運作方式 | 取捨 |
|---|---|---|
| **截斷** | 丟棄最舊的訊息 | 失去早期上下文 |
| **摘要** | 將較舊訊息濃縮成摘要 | 某些細節遺失，但保留重點 |
| **草稿板 / 外部記憶** | 將關鍵事實儲存在對話外部 | 需要工具呼叫，但可以存活任意縮減 |

在本筆記本中，我們結合了**摘要**與**草稿板工具**，讓代理即使對話歷史被濃縮，也能保持連貫性。


## 建立一個具備上下文感知能力的代理人


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模擬一段長對話

讓我們逐步進行一次多輪對話，看看上下文如何累積。代理人應該在對話過程中保留關鍵細節（偏好、預算、旅遊日期），並展示連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留先前回合的上下文 — 它知道關於日本、壽司、寺廟、攝影、$3000 預算、獨自旅行以及四月的時間範圍。在短對話中這運作良好，但隨著對話增長，重發完整歷史會變得昂貴。

讓我們繼續透過更多回合的對話來觀察上下文的累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話的增長，我們可以使用**摘要工具**將積累的上下文濃縮成緊湊的格式。代理會調用此工具來記錄關鍵偏好，以便即使較舊的訊息被刪除，重要資訊仍能被保存。

此模式是更高級歷史簡化的基礎構建塊：
1. 代理從對話中識別關鍵事實
2. 它調用摘要工具來保存這些事實
3. 由於摘要已捕捉到重要內容，可以安全刪除較舊的訊息

以下我們定義了一個代理可調用的`summarize_preferences`工具，用於記錄其所獲得的緊湊摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 摘要

在本課程中，你學會了如何使用 Microsoft Agent Framework 管理長時間運行的代理對話上下文：

### 主要概念
- **上下文視窗是有限的** — 對話歷史中的每個標記都會產生成本並計入限制。
- **摘要工具** 讓代理將累積的上下文濃縮成簡潔摘要，減少標記使用量，同時保留重要資訊。
- **代理草稿板** 提供持久的外部記憶，能在任何對話縮減後繼續保存。

### 你所建立的內容
- 一個**具上下文感知的代理**，可維持多回合對話的連續性
- 一個**摘要工具**（`summarize_preferences`），能以簡潔格式記錄關鍵用戶細節
- 一個**多回合對話**示範，上下文保留及變更處理

### 實際應用範疇
- **客戶服務機械人**：在長時間支援會話中記住偏好
- **個人助理**：追蹤持續進行的專案，避免重複解釋上下文
- **教育輔導**：在多次互動中維持學生進度

### 下一步
- 實作具檔案持久化的完整草稿板工具
- 加入摘要後自動截斷歷史
- 結合向量資料庫以進行語義記憶搜尋
- 建置能在日後重新以完整上下文繼續對話的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：  
本文件經由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。文件的原文版本應被視為權威來源。對於重要資料，建議採用專業人員翻譯。我們不會對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
